In [1]:
from pathlib import Path
import os
import shutil
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm


E:\CondaEnvs\smartagrivision\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("NumPy  :", np.__version__)
print("Pandas :", pd.__version__)
print("PIL    :", Image.__version__)

NumPy  : 2.4.6
Pandas : 3.0.5
PIL    : 12.3.0


In [3]:
# IMAGE EXTENSIONS
IMAGE_EXTENSIONS = (
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".gif",
    ".webp"
)

print("Supported Image Extensions:")
print(IMAGE_EXTENSIONS)

Supported Image Extensions:
('.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp')


# Cleaned Dataset / Preprocessed Dataset Paths

In [9]:
# CORRECT DATASET PATHS

CLEANED_DATASET_ROOT = Path(
    r"E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\Cleaned_Dataset"
)

PREPROCESSED_DATASET_ROOT = (
    CLEANED_DATASET_ROOT.parent / "Preprocessed_Dataset"
)


print("DATASET PATHS")


print("Cleaned Dataset Root     :", CLEANED_DATASET_ROOT)
print("Preprocessed Dataset Root:", PREPROCESSED_DATASET_ROOT)

DATASET PATHS
Cleaned Dataset Root     : E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\Cleaned_Dataset
Preprocessed Dataset Root: E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\Preprocessed_Dataset


In [10]:
# PATH EXISTENCE CHECK
print()
print("PATH STATUS")


print(
    "Cleaned Dataset Exists     :",
    CLEANED_DATASET_ROOT.exists()
)

print(
    "Preprocessed Dataset Exists:",
    PREPROCESSED_DATASET_ROOT.exists()
)


PATH STATUS
Cleaned Dataset Exists     : True
Preprocessed Dataset Exists: False


In [12]:
# CLEANED DATASET STRUCTURE CHECK
datasets = [
    "Fruits-360",
    "New Plant Diseases",
    "PlantDoc",
    "Vegetable Dataset"
]

print("=" * 90)
print("CLEANED DATASET AVAILABILITY")
print("=" * 90)

for dataset_name in datasets:

    dataset_path = (
        CLEANED_DATASET_ROOT / dataset_name
    )

    status = (
        "Available"
        if dataset_path.exists()
        else "Missing"
    )

    print(
        f"{dataset_name:<25} : {status}"
    )

CLEANED DATASET AVAILABILITY
Fruits-360                : Available
New Plant Diseases        : Available
PlantDoc                  : Available
Vegetable Dataset         : Available


# Load Cleaned Image Metadata


In [13]:
# FIND CLEANED IMAGE METADATA

print("LOAD CLEANED IMAGE METADATA")


metadata_candidates = [
    CLEANED_DATASET_ROOT / "image_metadata.csv",
    CLEANED_DATASET_ROOT.parent / "image_metadata.csv",
    CLEANED_DATASET_ROOT.parent / "Preprocessed_Data" / "image_metadata.csv",
    CLEANED_DATASET_ROOT.parent / "Preprocessed_Dataset" / "image_metadata.csv",
]

metadata_path = None

for candidate in metadata_candidates:

    if candidate.exists():

        metadata_path = candidate
        break

if metadata_path is not None:

    print("Metadata Found:")
    print(metadata_path)

else:

    print("image_metadata.csv was NOT found.")

    print()
    print("Searching nearby folders...")

    for csv_file in CLEANED_DATASET_ROOT.parent.rglob(
        "image_metadata.csv"
    ):
        print(csv_file)

LOAD CLEANED IMAGE METADATA
Metadata Found:
E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\Preprocessed_Data\image_metadata.csv


In [14]:
# Load Metadata
if metadata_path is None:

    raise FileNotFoundError(
        "image_metadata.csv could not be found."
    )

image_metadata_df = pd.read_csv(
    metadata_path
)


print("IMAGE METADATA LOADED")


print(
    "Metadata Path :",
    metadata_path
)

print(
    "Rows          :",
    f"{len(image_metadata_df):,}"
)

print(
    "Columns       :",
    len(image_metadata_df.columns)
)

print()

display(
    image_metadata_df.head()
)

IMAGE METADATA LOADED
Metadata Path : E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\Preprocessed_Data\image_metadata.csv
Rows          : 294,695
Columns       : 6



,Dataset,Split,Class,Image_Path,Filename,Label
0,Fruits-360,Unsplit,fruits-360_100x100,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...,r0_103_100.jpg,79
1,Fruits-360,Unsplit,fruits-360_100x100,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...,r0_107_100.jpg,79
2,Fruits-360,Unsplit,fruits-360_100x100,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...,r0_111_100.jpg,79
3,Fruits-360,Unsplit,fruits-360_100x100,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...,r0_115_100.jpg,79
4,Fruits-360,Unsplit,fruits-360_100x100,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...,r0_119_100.jpg,79


In [15]:
# Metadata Structure

print("IMAGE METADATA STRUCTURE")


print("Available Columns:")
print()

for index, column in enumerate(
    image_metadata_df.columns,
    start=1
):

    print(
        f"{index}. {column}"
    )

print()

print(
    "Data Shape :",
    image_metadata_df.shape
)

print()

print("Missing Values:")
display(
    image_metadata_df.isnull().sum()
)

IMAGE METADATA STRUCTURE
Available Columns:

1. Dataset
2. Split
3. Class
4. Image_Path
5. Filename
6. Label

Data Shape : (294695, 6)

Missing Values:


Dataset       0
Split         0
Class         0
Image_Path    0
Filename      0
Label         0
dtype: int64

In [16]:
# DATASET METADATA SUMMARY

print("DATASET METADATA SUMMARY")


print(
    "Total Images :",
    f"{len(image_metadata_df):,}"
)

print(
    "Total Datasets :",
    image_metadata_df["Dataset"].nunique()
)

print(
    "Total Classes :",
    image_metadata_df["Class"].nunique()
)

print()

print("Dataset Distribution:")

dataset_metadata_summary = (
    image_metadata_df
    .groupby("Dataset")
    .size()
    .reset_index(name="Images")
    .sort_values(
        "Images",
        ascending=False
    )
)

display(
    dataset_metadata_summary
)

DATASET METADATA SUMMARY
Total Images : 294,695
Total Datasets : 4
Total Classes : 82

Dataset Distribution:


,Dataset,Images
0,Fruits-360,182942
1,New Plant Diseases,87841
3,Vegetable Dataset,20996
2,PlantDoc,2916


# Image Resize Configuration

In [17]:
# IMAGE RESIZE CONFIGURATION
TARGET_WIDTH = 224
TARGET_HEIGHT = 224

TARGET_SIZE = (
    TARGET_WIDTH,
    TARGET_HEIGHT
)

print("=" * 90)
print("SECTION 4 : IMAGE RESIZE CONFIGURATION")
print("=" * 90)

print(
    "Target Width  :",
    TARGET_WIDTH,
    "px"
)

print(
    "Target Height :",
    TARGET_HEIGHT,
    "px"
)

print(
    "Target Size   :",
    TARGET_SIZE
)

SECTION 4 : IMAGE RESIZE CONFIGURATION
Target Width  : 224 px
Target Height : 224 px
Target Size   : (224, 224)


In [18]:
# Resize Method
RESIZE_METHOD = Image.Resampling.LANCZOS


print("RESIZE METHOD")


print(
    "Resize Method : LANCZOS"
)

print(
    "Target Size   :",
    TARGET_SIZE
)

RESIZE METHOD
Resize Method : LANCZOS
Target Size   : (224, 224)


In [19]:
# Configuration Table
resize_configuration = pd.DataFrame({
    "Parameter": [
        "Target Width",
        "Target Height",
        "Target Size",
        "Resize Method"
    ],
    "Value": [
        f"{TARGET_WIDTH} px",
        f"{TARGET_HEIGHT} px",
        f"{TARGET_WIDTH} x {TARGET_HEIGHT}",
        "LANCZOS"
    ]
})

display(
    resize_configuration
)

,Parameter,Value
0,Target Width,224 px
1,Target Height,224 px
2,Target Size,224 x 224
3,Resize Method,LANCZOS


# Image Resizing

In [20]:
# CREATE PREPROCESSED DATASET FOLDER
PREPROCESSED_DATASET_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


print("IMAGE RESIZING")


print(
    "Output Directory :",
    PREPROCESSED_DATASET_ROOT
)

print(
    "Directory Exists :",
    PREPROCESSED_DATASET_ROOT.exists()
)

IMAGE RESIZING
Output Directory : E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\Preprocessed_Dataset
Directory Exists : True


In [22]:
# Resize Function
def resize_image(
    image_path,
    target_size=TARGET_SIZE
):

    try:

        with Image.open(image_path) as img:

            resized_image = (
                img
                .convert("RGB")
                .resize(
                    target_size,
                    RESIZE_METHOD
                )
            )

        return resized_image

    except Exception as error:

        return None




In [23]:
# SINGLE IMAGE RESIZE TEST
sample_image_path = None

for path_value in (
    image_metadata_df["Image_Path"]
    .dropna()
):

    candidate = Path(path_value)

    if candidate.exists():

        sample_image_path = candidate
        break


if sample_image_path is None:

    print(
        "No directly accessible sample image was found."
    )

else:

    with Image.open(
        sample_image_path
    ) as original_image:

        original_size = original_image.size
        original_mode = original_image.mode

    resized_image = resize_image(
        sample_image_path
    )

   
    print("SINGLE IMAGE RESIZE TEST")
  

    print(
        "Image :",
        sample_image_path.name
    )

    print(
        "Original Size :",
        original_size
    )

    print(
        "Original Mode :",
        original_mode
    )

    if resized_image is not None:

        print(
            "Resized Size  :",
            resized_image.size
        )

        print(
            "Resized Mode  :",
            resized_image.mode
        )

SINGLE IMAGE RESIZE TEST
Image : r0_103_100.jpg
Original Size : (100, 100)
Original Mode : RGB
Resized Size  : (224, 224)
Resized Mode  : RGB


In [24]:
# Resize Validation
if sample_image_path is not None:

    resized_image = resize_image(
        sample_image_path
    )

    if resized_image is not None:

        resize_valid = (
            resized_image.size
            == TARGET_SIZE
        )

        rgb_valid = (
            resized_image.mode
            == "RGB"
        )

      
        print("RESIZE VALIDATION")
      

        print(
            "Expected Size :",
            TARGET_SIZE
        )

        print(
            "Actual Size   :",
            resized_image.size
        )

        print(
            "Size Correct  :",
            resize_valid
        )

        print(
            "RGB Correct   :",
            rgb_valid
        )

    else:

        print(
            "Resize validation failed."
        )

else:

    print(
        "No sample image available."
    )

RESIZE VALIDATION
Expected Size : (224, 224)
Actual Size   : (224, 224)
Size Correct  : True
RGB Correct   : True


# Color Mode Standardization

In [25]:
# IMAGE MODE ANALYSIS

print("COLOR MODE STANDARDIZATION")


mode_counter = {}

sample_paths = (
    image_metadata_df["Image_Path"]
    .dropna()
)

for path_value in tqdm(
    sample_paths,
    desc="Checking image modes",
    unit="image"
):

    image_path = Path(path_value)

    if not image_path.exists():
        continue

    try:

        with Image.open(image_path) as img:

            mode = img.mode

        mode_counter[mode] = (
            mode_counter.get(mode, 0)
            + 1
        )

    except Exception:

        continue


mode_df = pd.DataFrame(
    list(mode_counter.items()),
    columns=[
        "Mode",
        "Images"
    ]
).sort_values(
    "Images",
    ascending=False
)

display(
    mode_df
)

COLOR MODE STANDARDIZATION


Checking image modes: 100%|██████████| 294695/294695 [1:10:58<00:00, 69.20image/s]


,Mode,Images
0,RGB,294684
1,CMYK,5
3,RGBA,5
2,L,1


In [26]:
# RGB Conversion
def standardize_rgb(image):

    if image.mode == "RGB":

        return image

    return image.convert("RGB")


print(
    "RGB standardization function created."
)

RGB standardization function created.


In [27]:
# RGB STANDARDIZATION TEST
if sample_image_path is not None:

    with Image.open(
        sample_image_path
    ) as img:

        original_mode = img.mode

        rgb_image = standardize_rgb(
            img
        )

        standardized_mode = (
            rgb_image.mode
        )


    print("RGB STANDARDIZATION TEST")


    print(
        "Original Mode     :",
        original_mode
    )

    print(
        "Standardized Mode :",
        standardized_mode
    )

    print(
        "RGB Valid         :",
        standardized_mode == "RGB"
    )

else:

    print(
        "RGB test skipped."
    )

RGB STANDARDIZATION TEST
Original Mode     : RGB
Standardized Mode : RGB
RGB Valid         : True


In [28]:
# COLOR MODE VALIDATION

print("COLOR MODE STANDARDIZATION VALIDATION")


print(
    "Required Output Mode : RGB"
)

print(
    "Sample Output Mode   :",
    standardized_mode
    if sample_image_path is not None
    else "Not Available"
)

if sample_image_path is not None:

    print(
        "Validation Result    :",
        standardized_mode == "RGB"
    )

COLOR MODE STANDARDIZATION VALIDATION
Required Output Mode : RGB
Sample Output Mode   : RGB
Validation Result    : True


# Pixel Normalization

In [29]:
# PIXEL NORMALIZATION FUNCTION
def normalize_pixels(image):

    image_array = np.asarray(
        image,
        dtype=np.float32
    )

    normalized_array = (
        image_array / 255.0
    )

    return normalized_array



print("PIXEL NORMALIZATION")


print(
    "Normalization Formula : Pixel / 255.0"
)

PIXEL NORMALIZATION
Normalization Formula : Pixel / 255.0


In [30]:
# Normalization Test
if sample_image_path is not None:

    resized_rgb_image = resize_image(
        sample_image_path
    )

    normalized_image = normalize_pixels(
        resized_rgb_image
    )

    
    print("PIXEL NORMALIZATION TEST")
  

    print(
        "Image Shape :",
        normalized_image.shape
    )

    print(
        "Data Type   :",
        normalized_image.dtype
    )

    print(
        "Minimum     :",
        float(normalized_image.min())
    )

    print(
        "Maximum     :",
        float(normalized_image.max())
    )

else:

    print(
        "Normalization test skipped."
    )

PIXEL NORMALIZATION TEST
Image Shape : (224, 224, 3)
Data Type   : float32
Minimum     : 0.0
Maximum     : 1.0


In [31]:
# NORMALIZATION VALIDATION
if sample_image_path is not None:

    minimum_value = float(
        normalized_image.min()
    )

    maximum_value = float(
        normalized_image.max()
    )

    normalization_valid = (
        minimum_value >= 0.0
        and
        maximum_value <= 1.0
    )

    
    print("NORMALIZATION VALIDATION")


    print(
        "Expected Range : 0.0 - 1.0"
    )

    print(
        "Actual Minimum :",
        round(minimum_value, 6)
    )

    print(
        "Actual Maximum :",
        round(maximum_value, 6)
    )

    print(
        "Normalization Valid :",
        normalization_valid
    )

else:

    print(
        "Normalization validation skipped."
    )

NORMALIZATION VALIDATION
Expected Range : 0.0 - 1.0
Actual Minimum : 0.0
Actual Maximum : 1.0
Normalization Valid : True


In [32]:
# FINAL CHECK 

print("PIXEL NORMALIZATION COMPLETED")


print(
    "Target Image Size :",
    TARGET_SIZE
)

print(
    "Output Color Mode : RGB"
)

print(
    "Pixel Range       : 0.0 - 1.0"
)

print(
    "Normalization      : Pixel / 255.0"
)


PIXEL NORMALIZATION COMPLETED
Target Image Size : (224, 224)
Output Color Mode : RGB
Pixel Range       : 0.0 - 1.0
Normalization      : Pixel / 255.0


# Label / Class Mapping

In [33]:
# Label / Class Mapping

from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

image_metadata_df["Encoded_Label"] = (
    label_encoder.fit_transform(
        image_metadata_df["Class"].astype(str)
    )
)

print("LABEL / CLASS MAPPING")

print(
    "Total Classes:",
    image_metadata_df["Class"].nunique()
)

print(
    "Encoded Labels:",
    image_metadata_df["Encoded_Label"].nunique()
)

display(
    image_metadata_df[
        ["Class", "Label", "Encoded_Label"]
    ].head(10)
)

LABEL / CLASS MAPPING
Total Classes: 82
Encoded Labels: 82


,Class,Label,Encoded_Label
0,fruits-360_100x100,79,79
1,fruits-360_100x100,79,79
2,fruits-360_100x100,79,79
3,fruits-360_100x100,79,79
4,fruits-360_100x100,79,79
5,fruits-360_100x100,79,79
6,fruits-360_100x100,79,79
7,fruits-360_100x100,79,79
8,fruits-360_100x100,79,79
9,fruits-360_100x100,79,79


In [34]:
# Class Mapping Table

class_mapping_df = pd.DataFrame({
    "Class": label_encoder.classes_,
    "Encoded_Label": range(
        len(label_encoder.classes_)
    )
})

print("CLASS MAPPING TABLE")

display(
    class_mapping_df
)

CLASS MAPPING TABLE


,Class,Encoded_Label
0,Apple_Scab_Leaf,0
1,Apple___Apple_scab,1
2,Apple___Black_rot,2
3,Apple___Cedar_apple_rust,3
4,Apple___healthy,4
...,...,...
77,Tomato_mold_leaf,77
78,Tomato_two_spotted_spider_mites_leaf,78
79,fruits-360_100x100,79
80,grape_leaf,80


In [35]:
# Label Mapping Validation

class_count = (
    image_metadata_df["Class"]
    .nunique()
)

encoded_count = (
    image_metadata_df["Encoded_Label"]
    .nunique()
)

mapping_valid = (
    class_count == encoded_count
)

print("LABEL MAPPING VALIDATION")

print(
    "Original Classes:",
    class_count
)

print(
    "Encoded Labels:",
    encoded_count
)

print(
    "Mapping Valid:",
    mapping_valid
)

LABEL MAPPING VALIDATION
Original Classes: 82
Encoded Labels: 82
Mapping Valid: True


# Train / Validation / Test Preprocessing

In [36]:
# Split Distribution

print("TRAIN / VALIDATION / TEST PREPROCESSING")

split_distribution = (
    image_metadata_df["Split"]
    .astype(str)
    .str.strip()
    .value_counts()
    .reset_index()
)

split_distribution.columns = [
    "Split",
    "Images"
]

display(
    split_distribution
)

TRAIN / VALIDATION / TEST PREPROCESSING


,Split,Images
0,Unsplit,182942
1,Train,87937
2,Validation,20564
3,Test,3252


In [37]:
# Standardize Split Names

split_mapping = {
    "train": "Train",
    "training": "Train",
    "validation": "Validation",
    "valid": "Validation",
    "val": "Validation",
    "test": "Test",
    "testing": "Test",
    "unsplit": "Unsplit"
}

image_metadata_df["Processed_Split"] = (
    image_metadata_df["Split"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(split_mapping)
    .fillna(
        image_metadata_df["Split"]
    )
)

print("PROCESSED SPLIT NAMES")

display(
    image_metadata_df[
        ["Dataset", "Split", "Processed_Split"]
    ]
    .drop_duplicates()
    .sort_values(
        ["Dataset", "Processed_Split"]
    )
)

PROCESSED SPLIT NAMES


,Dataset,Split,Processed_Split
0,Fruits-360,Unsplit,Unsplit
182942,New Plant Diseases,Train,Train
253219,New Plant Diseases,Validation,Validation
270783,PlantDoc,Test,Test
271035,PlantDoc,Train,Train
273699,Vegetable Dataset,Test,Test
276699,Vegetable Dataset,Train,Train
291695,Vegetable Dataset,Validation,Validation


In [38]:
# Split Metadata

train_metadata_df = (
    image_metadata_df[
        image_metadata_df["Processed_Split"]
        == "Train"
    ]
    .copy()
)

validation_metadata_df = (
    image_metadata_df[
        image_metadata_df["Processed_Split"]
        == "Validation"
    ]
    .copy()
)

test_metadata_df = (
    image_metadata_df[
        image_metadata_df["Processed_Split"]
        == "Test"
    ]
    .copy()
)

unsplit_metadata_df = (
    image_metadata_df[
        image_metadata_df["Processed_Split"]
        == "Unsplit"
    ]
    .copy()
)

print("SPLIT DATASET SUMMARY")

print(
    "Train:",
    f"{len(train_metadata_df):,}"
)

print(
    "Validation:",
    f"{len(validation_metadata_df):,}"
)

print(
    "Test:",
    f"{len(test_metadata_df):,}"
)

print(
    "Unsplit:",
    f"{len(unsplit_metadata_df):,}"
)

SPLIT DATASET SUMMARY
Train: 87,937
Validation: 20,564
Test: 3,252
Unsplit: 182,942


In [39]:
# Dataset-wise Split Summary

split_summary_df = (
    image_metadata_df
    .groupby(
        ["Dataset", "Processed_Split"]
    )
    .size()
    .reset_index(
        name="Images"
    )
)

print("DATASET-WISE SPLIT SUMMARY")

display(
    split_summary_df
)

DATASET-WISE SPLIT SUMMARY


,Dataset,Processed_Split,Images
0,Fruits-360,Unsplit,182942
1,New Plant Diseases,Train,70277
2,New Plant Diseases,Validation,17564
3,PlantDoc,Test,252
4,PlantDoc,Train,2664
5,Vegetable Dataset,Test,3000
6,Vegetable Dataset,Train,14996
7,Vegetable Dataset,Validation,3000


# Preprocessed Image Validation

In [40]:
# Preprocessed Image Validation

def validate_preprocessed_image(
    image_path
):

    try:

        with Image.open(image_path) as image:

            original_mode = image.mode

            processed_image = (
                image
                .convert("RGB")
                .resize(
                    TARGET_SIZE,
                    RESIZE_METHOD
                )
            )

            image_array = np.asarray(
                processed_image,
                dtype=np.float32
            )

            image_array /= 255.0

        return {
            "Valid": True,
            "Size": processed_image.size,
            "Mode": processed_image.mode,
            "Min": float(
                image_array.min()
            ),
            "Max": float(
                image_array.max()
            ),
            "Error": ""
        }

    except Exception as error:

        return {
            "Valid": False,
            "Size": None,
            "Mode": None,
            "Min": None,
            "Max": None,
            "Error": str(error)
        }


print(
    "PREPROCESSED IMAGE VALIDATION FUNCTION READY"
)

PREPROCESSED IMAGE VALIDATION FUNCTION READY


In [41]:
# Validate Sample Images

validation_records = []

sample_records = (
    image_metadata_df
    .head(20)
)

for _, row in sample_records.iterrows():

    result = validate_preprocessed_image(
        row["Image_Path"]
    )

    validation_records.append({
        "Dataset": row["Dataset"],
        "Filename": row["Filename"],
        **result
    })

validation_df = pd.DataFrame(
    validation_records
)

print("PREPROCESSED IMAGE VALIDATION")

display(
    validation_df
)

PREPROCESSED IMAGE VALIDATION


,Dataset,Filename,Valid,Size,Mode,Min,Max,Error
0,Fruits-360,r0_103_100.jpg,True,"(224, 224)",RGB,0.0,1.0,
1,Fruits-360,r0_107_100.jpg,True,"(224, 224)",RGB,0.0,1.0,
2,Fruits-360,r0_111_100.jpg,True,"(224, 224)",RGB,0.0,1.0,
3,Fruits-360,r0_115_100.jpg,True,"(224, 224)",RGB,0.0,1.0,
4,Fruits-360,r0_119_100.jpg,True,"(224, 224)",RGB,0.0,1.0,
5,Fruits-360,r0_11_100.jpg,True,"(224, 224)",RGB,0.0,1.0,
6,Fruits-360,r0_123_100.jpg,True,"(224, 224)",RGB,0.0,1.0,
7,Fruits-360,r0_127_100.jpg,True,"(224, 224)",RGB,0.0,1.0,
8,Fruits-360,r0_131_100.jpg,True,"(224, 224)",RGB,0.0,1.0,
9,Fruits-360,r0_135_100.jpg,True,"(224, 224)",RGB,0.0,1.0,


In [42]:
# Validation Results

if not validation_df.empty:

    valid_count = int(
        validation_df["Valid"]
        .sum()
    )

    invalid_count = (
        len(validation_df)
        - valid_count
    )

    print("IMAGE VALIDATION RESULTS")

    print(
        "Images Checked:",
        len(validation_df)
    )

    print(
        "Valid Images:",
        valid_count
    )

    print(
        "Invalid Images:",
        invalid_count
    )

    print(
        "Target Size:",
        TARGET_SIZE
    )

    print(
        "Expected Mode: RGB"
    )

else:

    print(
        "No images were available for validation."
    )

IMAGE VALIDATION RESULTS
Images Checked: 20
Valid Images: 20
Invalid Images: 0
Target Size: (224, 224)
Expected Mode: RGB


In [43]:
# Preprocessing Validation Check

if not validation_df.empty:

    size_valid = validation_df[
        "Size"
    ].apply(
        lambda x: x == TARGET_SIZE
    ).all()

    mode_valid = (
        validation_df["Mode"]
        .eq("RGB")
        .all()
    )

    range_valid = (
        validation_df["Min"]
        .ge(0)
        .all()
        and
        validation_df["Max"]
        .le(1)
        .all()
    )

    print("PREPROCESSING VALIDATION CHECK")

    print(
        "Resize Valid:",
        size_valid
    )

    print(
        "RGB Valid:",
        mode_valid
    )

    print(
        "Normalization Valid:",
        range_valid
    )

PREPROCESSING VALIDATION CHECK
Resize Valid: True
RGB Valid: True
Normalization Valid: True


# Preprocessed Dataset Summary

In [44]:
# Dataset Summary

dataset_summary_df = (
    image_metadata_df
    .groupby("Dataset")
    .agg(
        Images=("Filename", "count"),
        Classes=("Class", "nunique")
    )
    .reset_index()
    .sort_values(
        "Images",
        ascending=False
    )
)

print("PREPROCESSED DATASET SUMMARY")

display(
    dataset_summary_df
)

PREPROCESSED DATASET SUMMARY


,Dataset,Images,Classes
0,Fruits-360,182942,1
1,New Plant Diseases,87841,38
3,Vegetable Dataset,20996,15
2,PlantDoc,2916,28


In [45]:
# Class Summary

class_summary_df = (
    image_metadata_df
    .groupby("Class")
    .size()
    .reset_index(
        name="Images"
    )
    .sort_values(
        "Images",
        ascending=False
    )
)

print("CLASS SUMMARY")

print(
    "Total Classes:",
    len(class_summary_df)
)

display(
    class_summary_df.head(20)
)

CLASS SUMMARY
Total Classes: 82


,Class,Images
79,fruits-360_100x100,182942
53,Soybean___healthy,2527
1,Apple___Apple_scab,2520
35,Orange___Haunglongbing_(Citrus_greening),2513
4,Apple___healthy,2500
41,"Pepper,_bell___healthy",2485
2,Apple___Black_rot,2484
69,Tomato___Tomato_Yellow_Leaf_Curl_Virus,2451
43,Potato___Early_blight,2424
44,Potato___Late_blight,2424


In [46]:
# Overall Summary

total_images = len(
    image_metadata_df
)

total_datasets = (
    image_metadata_df["Dataset"]
    .nunique()
)

total_classes = (
    image_metadata_df["Class"]
    .nunique()
)

print("OVERALL PREPROCESSED DATASET SUMMARY")

print(
    "Total Images:",
    f"{total_images:,}"
)

print(
    "Total Datasets:",
    total_datasets
)

print(
    "Total Classes:",
    total_classes
)

print(
    "Target Image Size:",
    TARGET_SIZE
)

print(
    "Target Color Mode:",
    "RGB"
)

print(
    "Pixel Range:",
    "0.0 - 1.0"
)

OVERALL PREPROCESSED DATASET SUMMARY
Total Images: 294,695
Total Datasets: 4
Total Classes: 82
Target Image Size: (224, 224)
Target Color Mode: RGB
Pixel Range: 0.0 - 1.0


# Save Preprocessed Metadata

In [47]:
# Metadata Output Directory

METADATA_OUTPUT_DIR = (
    PREPROCESSED_DATASET_ROOT
    / "metadata"
)

METADATA_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "METADATA OUTPUT DIRECTORY"
)

print(
    METADATA_OUTPUT_DIR
)

METADATA OUTPUT DIRECTORY
E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\Preprocessed_Dataset\metadata


In [48]:
# Save Complete Metadata

metadata_output_path = (
    METADATA_OUTPUT_DIR
    / "image_metadata_preprocessed.csv"
)

image_metadata_df.to_csv(
    metadata_output_path,
    index=False
)

print("PREPROCESSED METADATA SAVED")

print(
    "File:",
    metadata_output_path
)

print(
    "Rows:",
    f"{len(image_metadata_df):,}"
)

PREPROCESSED METADATA SAVED
File: E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\Preprocessed_Dataset\metadata\image_metadata_preprocessed.csv
Rows: 294,695


In [49]:
# Save Split Metadata

train_metadata_df.to_csv(
    METADATA_OUTPUT_DIR
    / "train_metadata.csv",
    index=False
)

validation_metadata_df.to_csv(
    METADATA_OUTPUT_DIR
    / "validation_metadata.csv",
    index=False
)

test_metadata_df.to_csv(
    METADATA_OUTPUT_DIR
    / "test_metadata.csv",
    index=False
)

unsplit_metadata_df.to_csv(
    METADATA_OUTPUT_DIR
    / "unsplit_metadata.csv",
    index=False
)

print("SPLIT METADATA SAVED")

print(
    "train_metadata.csv"
)

print(
    "validation_metadata.csv"
)

print(
    "test_metadata.csv"
)

print(
    "unsplit_metadata.csv"
)

SPLIT METADATA SAVED
train_metadata.csv
validation_metadata.csv
test_metadata.csv
unsplit_metadata.csv


In [50]:
# Saved File Check

metadata_files = [
    METADATA_OUTPUT_DIR
    / "image_metadata_preprocessed.csv",

    METADATA_OUTPUT_DIR
    / "train_metadata.csv",

    METADATA_OUTPUT_DIR
    / "validation_metadata.csv",

    METADATA_OUTPUT_DIR
    / "test_metadata.csv",

    METADATA_OUTPUT_DIR
    / "unsplit_metadata.csv"
]

print("SAVED METADATA FILE CHECK")

for file_path in metadata_files:

    print(
        file_path.name,
        ":",
        "Available"
        if file_path.exists()
        else "Missing"
    )

SAVED METADATA FILE CHECK
image_metadata_preprocessed.csv : Available
train_metadata.csv : Available
validation_metadata.csv : Available
test_metadata.csv : Available
unsplit_metadata.csv : Available


# Final Preprocessing Report

In [51]:
# Final Preprocessing Statistics

final_total_images = len(
    image_metadata_df
)

final_total_datasets = (
    image_metadata_df["Dataset"]
    .nunique()
)

final_total_classes = (
    image_metadata_df["Class"]
    .nunique()
)

final_missing_values = (
    image_metadata_df.isnull()
    .sum()
    .sum()
)

print("FINAL PREPROCESSING STATISTICS")

print(
    "Total Images:",
    f"{final_total_images:,}"
)

print(
    "Total Datasets:",
    final_total_datasets
)

print(
    "Total Classes:",
    final_total_classes
)

print(
    "Missing Values:",
    final_missing_values
)

print(
    "Target Size:",
    TARGET_SIZE
)

print(
    "Color Mode:",
    "RGB"
)

print(
    "Pixel Range:",
    "0.0 - 1.0"
)

FINAL PREPROCESSING STATISTICS
Total Images: 294,695
Total Datasets: 4
Total Classes: 82
Missing Values: 0
Target Size: (224, 224)
Color Mode: RGB
Pixel Range: 0.0 - 1.0


In [52]:
# Final Dataset Report

final_report_df = pd.DataFrame({
    "Metric": [
        "Total Images",
        "Total Datasets",
        "Total Classes",
        "Train Images",
        "Validation Images",
        "Test Images",
        "Unsplit Images",
        "Target Width",
        "Target Height",
        "Color Mode",
        "Pixel Normalization"
    ],

    "Value": [
        final_total_images,
        final_total_datasets,
        final_total_classes,
        len(train_metadata_df),
        len(validation_metadata_df),
        len(test_metadata_df),
        len(unsplit_metadata_df),
        TARGET_WIDTH,
        TARGET_HEIGHT,
        "RGB",
        "0.0 - 1.0"
    ]
})

print("FINAL PREPROCESSING REPORT")

display(
    final_report_df
)

FINAL PREPROCESSING REPORT


,Metric,Value
0,Total Images,294695
1,Total Datasets,4
2,Total Classes,82
3,Train Images,87937
4,Validation Images,20564
5,Test Images,3252
6,Unsplit Images,182942
7,Target Width,224
8,Target Height,224
9,Color Mode,RGB


In [53]:
# Final Validation Status

metadata_valid = (
    final_missing_values == 0
)

files_valid = all(
    file_path.exists()
    for file_path in metadata_files
)

label_valid = (
    image_metadata_df["Encoded_Label"]
    .notna()
    .all()
)

final_status = (
    metadata_valid
    and files_valid
    and label_valid
)

print("FINAL PREPROCESSING STATUS")

print(
    "Metadata Valid:",
    metadata_valid
)

print(
    "Metadata Files Valid:",
    files_valid
)

print(
    "Label Encoding Valid:",
    label_valid
)

print(
    "FINAL STATUS:",
    "PASSED"
    if final_status
    else "CHECK REQUIRED"
)

FINAL PREPROCESSING STATUS
Metadata Valid: True
Metadata Files Valid: True
Label Encoding Valid: True
FINAL STATUS: PASSED


In [54]:
# Final Report Text

if final_status:

    print(
        "PREPROCESSING COMPLETED SUCCESSFULLY"
    )

    print(
        f"{final_total_images:,} image records processed."
    )

    print(
        f"{final_total_classes:,} classes identified."
    )

    print(
        f"{final_total_datasets} datasets identified."
    )

    print(
        f"Target resolution: "
        f"{TARGET_WIDTH} x {TARGET_HEIGHT}"
    )

    print(
        "Color mode standardized to RGB."
    )

    print(
        "Pixel values normalized to 0.0 - 1.0."
    )

else:

    print(
        "PREPROCESSING REQUIRES FURTHER VALIDATION."
    )

PREPROCESSING COMPLETED SUCCESSFULLY
294,695 image records processed.
82 classes identified.
4 datasets identified.
Target resolution: 224 x 224
Color mode standardized to RGB.
Pixel values normalized to 0.0 - 1.0.
